# Document AI 핸즈온: GTIN/UPC 추출 및 상품 정보 기록

<a href="https://colab.research.google.com/github/cjk0604/ocr-with-gemini/blob/main/ocr_with_document_ai.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab">
</a>
<a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fcjk0604%2Focr-with-gemini%2Fmain%2Focr_with_document_ai.ipynb">
  <img src="https://img.shields.io/badge/Colab_Enterprise-Open-blue?logo=google-cloud" alt="Open In Colab Enterprise">
</a>
<a href="https://github.com/cjk0604/ocr-with-gemini/blob/main/ocr_with_document_ai.ipynb">
  <img src="https://img.shields.io/badge/GitHub-View_Source-black?logo=github" alt="View on GitHub">
</a>

Google Cloud Document AI의 **OCR Processor**와 **Layout Parser**를 활용하여
상품 이미지/문서에서 바코드(GTIN, UPC, EAN 등)를 감지하고, 텍스트를 추출하며,
Layout Parser의 **Chunking** 기능으로 문서를 의미 단위로 분할하는 핸즈온 노트북입니다.

## 다른 노트북과의 차이점

| 항목 | Gemini OCR | Cloud Vision AI | **Document AI** |
|------|-----------|-----------------|-----------------|
| 방식 | 생성형 AI (LLM) | CV + ML 파이프라인 | **문서 특화 AI 파이프라인** |
| OCR | 프롬프트 기반 | `TEXT_DETECTION` | **OCR Processor** |
| 바코드 감지 | LLM이 해석 | OCR 텍스트 + 정규식 | **`detected_barcodes` 네이티브 지원** |
| 문서 구조 | 프롬프트로 추론 | 미지원 | **Layout Parser + Chunking** |
| 입력 형식 | 이미지 | 이미지 | **이미지 + PDF + TIFF 등** |
| 강점 | 유연한 추론 | 범용 이미지 분석 | **문서/바코드 특화, 구조화된 출력** |

## 목차
1. Setup
2. 공통 함수 정의
3. OCR Processor — 텍스트 및 바코드 추출
4. Layout Parser — 문서 구조 분석 및 Chunking
5. GTIN/UPC 추출 및 상품 정보 기록

---
## 1. Setup

필요한 라이브러리를 설치하고 Document AI 클라이언트를 초기화합니다.

| 패키지 | 역할 |
|--------|------|
| `google-cloud-documentai` | Document AI API 클라이언트. OCR, Layout Parser, 바코드 감지 등 문서 처리 기능을 제공합니다. |
| `opencv-python` | 이미지 읽기, 색상 변환, bounding box 그리기 등 컴퓨터 비전 처리에 사용합니다. |
| `Pillow` | Python 이미지 처리 라이브러리(PIL). 이미지를 열고, 변환하고, 노트북에서 출력하는 데 사용합니다. |

### Document AI Processor 준비

이 노트북을 실행하려면 Google Cloud Console에서 **Document AI Processor**를 미리 생성해야 합니다:

1. [Document AI Console](https://console.cloud.google.com/ai/document-ai)로 이동
2. **프로세서 만들기** 클릭
3. 아래 프로세서를 각각 생성:
   - **OCR** (`OCR_PROCESSOR`) — 텍스트 및 바코드 추출용
   - **Layout Parser** (`LAYOUT_PARSER_PROCESSOR`) — 문서 구조 분석 및 Chunking용
4. 생성된 프로세서의 **ID**를 아래 셀에 입력

In [ ]:
!pip install -q google-cloud-documentai opencv-python Pillow

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import requests
from google.api_core.client_options import ClientOptions
from google.cloud import documentai_v1 as documentai
from PIL import Image

In [ ]:
# TODO: 본인의 Google Cloud 프로젝트 정보 입력
PROJECT_ID = ""           # Google Cloud 프로젝트 ID
LOCATION = "us"           # 프로세서 위치 (us 또는 eu)
OCR_PROCESSOR_ID = ""     # OCR 프로세서 ID
LAYOUT_PROCESSOR_ID = ""  # Layout Parser 프로세서 ID

# Document AI 클라이언트 초기화
opts = ClientOptions(api_endpoint=f"{LOCATION}-documentai.googleapis.com")
client = documentai.DocumentProcessorServiceClient(client_options=opts)

---
## 2. 공통 함수 정의

In [ ]:
BOX_COLORS = [
    (255, 56, 56), (255, 157, 151), (255, 112, 31), (255, 178, 29),
    (207, 210, 49), (72, 249, 10), (146, 204, 23), (61, 219, 134),
    (26, 147, 52), (0, 212, 187), (44, 153, 168), (0, 194, 255),
    (52, 69, 147), (100, 115, 255), (0, 24, 236), (132, 56, 255),
    (82, 0, 133), (203, 56, 255), (255, 149, 200), (255, 55, 199),
]


def download_image(url, filename):
    """URL에서 이미지를 다운로드합니다."""
    path = Path(filename)
    if not path.exists():
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        path.write_bytes(resp.content)
    return filename


def read_image(filename):
    """이미지를 읽어 PIL Image와 너비/높이를 반환합니다."""
    image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    return Image.fromarray(image), w, h


def process_document(processor_id, file_path, mime_type="image/jpeg", process_options=None):
    """Document AI 프로세서로 문서를 처리합니다."""
    resource_name = client.processor_path(PROJECT_ID, LOCATION, processor_id)
    raw_document = documentai.RawDocument(
        content=Path(file_path).read_bytes(),
        mime_type=mime_type,
    )
    request = documentai.ProcessRequest(
        name=resource_name,
        raw_document=raw_document,
    )
    if process_options:
        request.process_options = process_options

    result = client.process_document(request=request)
    return result.document


def get_text_from_layout(layout, full_text):
    """Document AI Layout 객체에서 텍스트를 추출합니다."""
    response = ""
    for segment in layout.text_anchor.text_segments:
        start = int(segment.start_index)
        end = int(segment.end_index)
        response += full_text[start:end]
    return response


def draw_doc_boxes(pil_image, layouts, labels, w, h):
    """Document AI의 bounding_poly (normalized vertices)를 이미지 위에 그립니다."""
    img = np.array(pil_image)
    for idx, (layout, label) in enumerate(zip(layouts, labels)):
        vertices = layout.bounding_poly.normalized_vertices
        pts = np.array([(int(v.x * w), int(v.y * h)) for v in vertices], dtype=np.int32)
        color = BOX_COLORS[idx % len(BOX_COLORS)]
        cv2.polylines(img, [pts], True, color, 2)

        x, y = pts[0]
        display_label = label[:50]
        (tw, th), _ = cv2.getTextSize(display_label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(img, (x, y - th - 6), (x + tw, y), color, -1)
        cv2.putText(img, display_label, (x, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

    return Image.fromarray(img)

---
## 3. OCR Processor — 텍스트 및 바코드 추출

Document AI의 OCR Processor는 이미지/PDF에서 텍스트를 추출하면서 동시에 **바코드도 네이티브로 감지**합니다.
Cloud Vision의 `TEXT_DETECTION`과 달리, 정규식 파싱 없이도 바코드 유형과 값을 직접 반환합니다.

### 지원하는 바코드 형식
`QR_CODE`, `EAN_13`, `EAN_8`, `UPC_A`, `UPC_E`, `CODE_128`, `CODE_39`, `ITF`, `CODABAR`, `DATA_MATRIX`, `PDF_417` 등

In [ ]:
# 바코드 테스트 이미지 다운로드 (원하는 상품 바코드 이미지로 교체 가능)
download_image(
    "https://free-barcode.com/barcode/barcode-technology/detail-gtin-14-barcode/5.jpg",
    "barcode_image.jpg",
)

ocr_image, ocr_w, ocr_h = read_image("barcode_image.jpg")
display(ocr_image)

### 3-1. OCR — 전체 텍스트 추출

OCR Processor로 이미지를 처리하고, 감지된 전체 텍스트와 페이지별 블록/단락/줄 구조를 확인합니다.

In [ ]:
# OCR Processor로 이미지 처리
document = process_document(OCR_PROCESSOR_ID, "barcode_image.jpg")

print("=== 전체 OCR 텍스트 ===\n")
print(document.text)

print(f"\n=== 문서 구조 ===")
print(f"  페이지 수: {len(document.pages)}")
for page_idx, page in enumerate(document.pages):
    print(f"\n  --- 페이지 {page_idx + 1} ---")
    print(f"  블록(Block) 수: {len(page.blocks)}")
    print(f"  단락(Paragraph) 수: {len(page.paragraphs)}")
    print(f"  줄(Line) 수: {len(page.lines)}")
    print(f"  토큰(Token) 수: {len(page.tokens)}")
    print(f"  감지된 바코드 수: {len(page.detected_barcodes)}")

### 3-2. 텍스트 블록 시각화

감지된 줄(Line) 단위로 bounding box를 그려 OCR 결과를 시각적으로 확인합니다.

In [ ]:
page = document.pages[0]

# 줄(Line) 단위 bounding box 시각화
layouts = []
labels = []
for line in page.lines:
    line_text = get_text_from_layout(line.layout, document.text).strip()
    layouts.append(line.layout)
    labels.append(line_text)
    print(f"  \"{line_text}\"  신뢰도: {line.layout.confidence:.1%}")

display(draw_doc_boxes(ocr_image, layouts, labels, ocr_w, ocr_h))

### 3-3. 바코드 네이티브 감지

Document AI는 OCR과 동시에 바코드를 **네이티브로 감지**합니다.
정규식 파싱 없이 바코드 형식(`format_`)과 값(`raw_value`)을 직접 반환합니다.

In [ ]:
barcode_layouts = []
barcode_labels = []

for page in document.pages:
    if page.detected_barcodes:
        print("=== 감지된 바코드 ===\n")
        for barcode_obj in page.detected_barcodes:
            barcode = barcode_obj.barcode
            print(f"  형식: {barcode.format_}")
            print(f"  값:   {barcode.raw_value}")
            print(f"  값 형식: {barcode.value_format}")
            print()

            barcode_layouts.append(barcode_obj.layout)
            barcode_labels.append(f"{barcode.format_}: {barcode.raw_value}")
    else:
        print("바코드가 감지되지 않았습니다.")

if barcode_layouts:
    display(draw_doc_boxes(ocr_image, barcode_layouts, barcode_labels, ocr_w, ocr_h))

---
## 4. Layout Parser — 문서 구조 분석 및 Chunking

Layout Parser는 문서의 **논리적 구조**(제목, 단락, 표, 목록 등)를 파악하고,
**Chunking** 기능으로 문서를 의미 단위의 청크로 분할합니다.

이 기능은 RAG(Retrieval-Augmented Generation) 파이프라인에서
문서를 벡터 DB에 저장할 때 특히 유용합니다.

### Chunking 파라미터

| 파라미터 | 설명 |
|----------|------|
| `chunk_size` | 각 청크의 최대 크기 (문자 수 기준) |
| `include_ancestor_headings` | 상위 제목을 청크에 포함할지 여부 |

### 4-1. Layout Parser로 문서 처리

Layout Parser는 이미지뿐 아니라 PDF 문서에서도 구조를 분석할 수 있습니다.
여기서는 동일한 바코드 이미지를 Layout Parser로 처리하여 결과를 비교합니다.

In [ ]:
# Layout Parser (Chunking 없이) — 문서 구조 분석
layout_document = process_document(LAYOUT_PROCESSOR_ID, "barcode_image.jpg")

print("=== Layout Parser — 전체 텍스트 ===\n")
print(layout_document.text)

print(f"\n=== 문서 구조 ===")
for page_idx, page in enumerate(layout_document.pages):
    print(f"\n  --- 페이지 {page_idx + 1} ---")
    print(f"  블록 수: {len(page.blocks)}")
    print(f"  단락 수: {len(page.paragraphs)}")
    print(f"  줄 수: {len(page.lines)}")
    print(f"  바코드 수: {len(page.detected_barcodes)}")

### 4-2. Layout Parser + Chunking

`ProcessOptions`에 `LayoutConfig.ChunkingConfig`를 설정하여
문서를 의미 단위의 청크로 자동 분할합니다.

In [ ]:
# Chunking 옵션 설정
chunking_config = documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
    chunk_size=500,
    include_ancestor_headings=True,
)

process_options = documentai.ProcessOptions(
    layout_config=documentai.ProcessOptions.LayoutConfig(
        chunking_config=chunking_config,
    )
)

# Layout Parser + Chunking으로 처리
chunked_document = process_document(
    LAYOUT_PROCESSOR_ID,
    "barcode_image.jpg",
    process_options=process_options,
)

print("=== Chunking 결과 ===\n")
if chunked_document.chunked_document and chunked_document.chunked_document.chunks:
    chunks = chunked_document.chunked_document.chunks
    print(f"총 청크 수: {len(chunks)}\n")

    for idx, chunk in enumerate(chunks):
        print(f"--- 청크 {idx + 1} ---")
        print(f"  청크 ID: {chunk.chunk_id}")
        print(f"  내용:\n{chunk.content}\n")

        if chunk.page_headers:
            print(f"  상위 제목:")
            for header in chunk.page_headers:
                print(f"    - {header.text}")
        print()
else:
    print("청크가 생성되지 않았습니다. (이미지의 텍스트가 적을 수 있습니다)")
    print("PDF 문서를 사용하면 더 많은 청크가 생성됩니다.")

### 4-3. PDF 문서로 Chunking 테스트

Layout Parser의 Chunking은 텍스트가 많은 **PDF 문서**에서 진가를 발휘합니다.
아래 셀에서 PDF 파일 경로를 지정하여 테스트해 보세요.

In [ ]:
# pdf_path = "your_document.pdf"  # TODO: 본인의 PDF 경로로 변경
#
# chunking_config = documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
#     chunk_size=500,
#     include_ancestor_headings=True,
# )
# process_options = documentai.ProcessOptions(
#     layout_config=documentai.ProcessOptions.LayoutConfig(
#         chunking_config=chunking_config,
#     )
# )
#
# pdf_document = process_document(
#     LAYOUT_PROCESSOR_ID,
#     pdf_path,
#     mime_type="application/pdf",
#     process_options=process_options,
# )
#
# print(f"전체 텍스트 길이: {len(pdf_document.text)} 문자")
# print(f"페이지 수: {len(pdf_document.pages)}")
#
# if pdf_document.chunked_document and pdf_document.chunked_document.chunks:
#     chunks = pdf_document.chunked_document.chunks
#     print(f"총 청크 수: {len(chunks)}\n")
#     for idx, chunk in enumerate(chunks):
#         print(f"--- 청크 {idx + 1} (ID: {chunk.chunk_id}) ---")
#         print(f"{chunk.content[:300]}...")
#         print()

---
## 5. GTIN/UPC 추출 및 상품 정보 기록

OCR Processor의 `detected_barcodes`와 OCR 텍스트를 결합하여
상품 정보를 구조화된 JSON으로 기록합니다.

Document AI의 장점: 바코드 형식과 값이 **API 응답에 직접 포함**되므로
정규식 파싱이 필요 없습니다.

In [ ]:
def analyze_product_docai(image_path, mime_type="image/jpeg"):
    """Document AI OCR Processor로 상품 정보를 종합 분석합니다."""
    doc = process_document(OCR_PROCESSOR_ID, image_path, mime_type=mime_type)

    # 바코드 추출 (네이티브)
    barcodes = []
    for page in doc.pages:
        for barcode_obj in page.detected_barcodes:
            barcode = barcode_obj.barcode
            barcodes.append({
                "format": barcode.format_,
                "raw_value": barcode.raw_value,
                "value_format": barcode.value_format,
            })

    # 페이지별 블록 텍스트 추출
    blocks_text = []
    for page in doc.pages:
        for block in page.blocks:
            text = get_text_from_layout(block.layout, doc.text).strip()
            if text:
                blocks_text.append({
                    "text": text,
                    "confidence": round(block.layout.confidence, 3),
                })

    product_info = {
        "image_path": image_path,
        "barcodes": barcodes,
        "ocr_full_text": doc.text.strip(),
        "text_blocks": blocks_text,
        "page_count": len(doc.pages),
    }

    return product_info


product_info = analyze_product_docai("barcode_image.jpg")
print(json.dumps(product_info, indent=2, ensure_ascii=False))

### 5-1. 나만의 상품 이미지로 테스트

아래 셀에서 `image_path`를 변경하여 본인의 상품 이미지를 테스트해 보세요.

In [ ]:
# image_path = "your_product_image.jpg"  # TODO: 본인의 이미지 경로로 변경
# my_image, w, h = read_image(image_path)
# display(my_image)
#
# # 종합 분석
# print("=== 상품 종합 분석 (Document AI) ===")
# product_info = analyze_product_docai(image_path)
# print(json.dumps(product_info, indent=2, ensure_ascii=False))
#
# # OCR 텍스트 영역 시각화
# doc = process_document(OCR_PROCESSOR_ID, image_path)
# layouts = [line.layout for page in doc.pages for line in page.lines]
# labels = [get_text_from_layout(l, doc.text).strip() for l in layouts]
# display(draw_doc_boxes(my_image, layouts, labels, w, h))